# Lesson 7.2 · Complexity, overfitting and pruning

A tree without limits can bring the training accuracy to 1.0 on any dataset without duplicates with different labels: it is a model of **unlimited complexity**. This lesson shows how to control it and why trees are **high-variance** models.

**What you will learn**

- The three ways to limit a tree: `max_depth`, `min_samples_leaf`, `ccp_alpha` (cost-complexity pruning).
- How overfitting shows up in a validation curve.
- Why small changes in the data change the tree (instability).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, validation_curve
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(1500, n_features=10, n_informative=4, n_redundant=0, class_sep=0.8, flip_y=0.1, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
tr, va = validation_curve(DecisionTreeClassifier(random_state=0), X_tr, y_tr, param_name="max_depth", param_range=depths, cv=cv)
tab = pd.DataFrame({"train": tr.mean(1), "validation": va.mean(1)}, index=[str(p) for p in depths])
print(tab.round(3).T)
full = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print("tree without limits: depth", full.get_depth(), "| leaves", full.get_n_leaves(), "| train", full.score(X_tr, y_tr), "| test", round(full.score(X_te, y_te), 3))

## Three knobs

- **`max_depth`**: limits the number of questions in a row. Simple and readable, but arbitrary (the right depth depends on the branch).
- **`min_samples_leaf`** (and `min_samples_split`): prevents leaves with few examples. It directly controls the **stability** of the leaf predictions: with 20 examples per leaf a probability is much less noisy.
- **`ccp_alpha`**: *cost-complexity pruning*. The full tree is built and then the branches that are not "worth" their complexity are removed: each leaf costs $\alpha$, and a branch is cut if the impurity gain it brings is smaller than the cost of its leaves.

In [ ]:
full_tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
path = full_tree.cost_complexity_pruning_path(X_tr, y_tr)
print("candidate alpha values:", len(path.ccp_alphas))
chosen = path.ccp_alphas[::max(1, len(path.ccp_alphas) // 8)]
for a in chosen:
    t = DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(X_tr, y_tr)
    print(f"alpha {a:.5f} | leaves {t.get_n_leaves():4d} | train {t.score(X_tr, y_tr):.3f} | test {t.score(X_te, y_te):.3f}")

With `alpha = 0` the tree is complete; as `alpha` grows the leaves shrink in number and the gap between train and test closes, down to a tiny tree. Here too the best value is chosen with cross-validation, and the rule "the simplest within one standard error of the best" is worth using.

## Unstable trees

A tree makes cascading choices: if the split at the root changes (because two features have almost the same gain), **everything else changes**. Resampling the data slightly, the tree can have a different structure and different predictions on many examples. Here the root is stable (the same feature in 29 resamples out of 30), but two bootstrap trees of depth 5 still disagree on 15% of the test examples. This is the **high variance**: the reason forests exist.

In [ ]:
rng = np.random.default_rng(0)
roots, predictions = [], []
for _ in range(30):
    i = rng.integers(0, len(X_tr), len(X_tr))                          # bootstrap resample
    t = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr[i], y_tr[i])
    roots.append(t.tree_.feature[0]); predictions.append(t.predict(X_te))
predictions = np.array(predictions)
print("feature chosen at the root in the 30 resamples:", np.bincount(roots, minlength=10))
print("average fraction of test examples on which two bootstrap trees disagree:", round(np.mean([(predictions[i] != predictions[j]).mean() for i in range(10) for j in range(i + 1, 10)]), 3))

> **Common mistake.** Choosing `max_depth` by hand by looking at a single tree: it is an unstable model, and the same choice on another sample can give a different structure. Always validate with cross-validation and do not interpret the details of a single tree (precise thresholds, order of the questions) as "truth".

## Check your understanding

<details><summary>What changes between `max_depth` and `min_samples_leaf` as a way of regularizing?</summary>

`max_depth` limits the complexity uniformly (all the branches reach at most that depth); `min_samples_leaf` limits it adaptively: a branch in a dense region can go down a lot, one in a sparse region stops early. Usually `min_samples_leaf` is more effective because it ties the complexity to the amount of data.

</details>

<details><summary>Why is cost-complexity pruning preferable to stopping the growth early?</summary>

Because stopping early (pre-pruning) is shortsighted: a split that is of little use now can enable very useful ones later. Post-pruning builds the full tree and cuts the branches that are not worth it, looking at the overall effect of the branch.

</details>